<a href="https://colab.research.google.com/github/wmjx691/rental-market-analyzer/blob/main/scraper_withGeo_Fencing_asset_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### 第一步：環境安裝（Cell 1）

Colab 是 Linux 環境，沒有視窗介面，所以我們必須安裝「無頭模式（Headless）」的瀏覽器驅動。

In [ ]:
# @title 1. 安裝必要套件、瀏覽器驅動與中文字型 (修復方塊字版)
# 安裝 selenium 和 google drive 相關套件
!pip install selenium gspread oauth2client webdriver_manager

# --- 新增：安裝地理資訊處理套件 ---
!pip install geopy

# 1. 安裝 Google Chrome
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f -y

# 2. 關鍵修正：安裝中文字型 (解決截圖方塊字問題)
!apt-get install -y fonts-noto-cjk

print("✅ 環境安裝完成！Geopy 已就緒。")

#### 第二步：Google 權限驗證（Cell 2）

這是 Colab 最強大的地方，不用搞複雜的 API Key，直接用你的 Google 帳號登入驗證，就能讓程式控制你的試算表。
*執行時會跳出視窗要求權限，請點選「允許」。*

In [ ]:
# @title 2. Google 帳號授權與試算表連線
from google.colab import auth
import gspread
from google.auth import default

# 進行身分驗證
auth.authenticate_user()       # 這一行會跳出彈窗要你登入
creds, _ = default()           # 這是暫時性的 Session 憑證
gc = gspread.authorize(creds)

print("Google 帳號授權成功！準備開始爬蟲...")

#### 第三步：爬蟲主程式（Cell 3）

這段程式碼會做兩件事：

1. **爬取目標租屋網**（使用無頭模式，因為 Colab 看不到畫面）。
2. **寫入試算表**：如果檔案不存在，它會自動建立一個名為 `TARGET_SHEET_NAME_advanced_withdistance` 的試算表；如果存在，它會把新資料「附加」在最後面。

In [ ]:
# @title 3. 初始化：全域設定、函式定義與載入資料庫 (Run Once)
import time
import pandas as pd
import re
import pytz
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from IPython.display import Image, display
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

# ==========================================
# ⚙️ 全域設定區 (CONFIG) - 修改這裡即可改變爬蟲目標
# ==========================================

# 1. 檔案名稱設定
SHEET_NAME_NEW = 'TARGET_SHEET_NAME_advanced_withdistance'
SHEET_NAME_OLD = 'TARGET_SHEET_NAME_advanced_copy'

# 2. 地理位置與目標設定
TARGET_CITY = "高雄市"          # 目標縣市 (用於地址解析與補全)
TARGET_REGION_CODE = "17"      # 地區代碼 (1=台北, 3=新北, 17=高雄, 15=台南...)
TARGET_DISTRICTS = ["***HIDDEN_District***", "***HIDDEN_District***", "鳳山區"] # 目標行政區列表 (可無限新增)
RENTAL_TYPE = "整層住家"        # 租屋類型

# 3. 參考點設定 (Anchor)
ANCHOR_NAME = "正修科技大學"
ANCHOR_COORDS = (22.6486, 120.35)

# 4. 其他系統設定
TW_TZ = pytz.timezone('Asia/Taipei')

# ==========================================

# --- 1. 設定瀏覽器選項 ---
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# --- 2. 工具函式定義 ---

def click_element_by_text(driver, text):
    try:
        xpath = f"//label[contains(text(),'{text}')] | //span[contains(text(),'{text}')] | //li[contains(text(),'{text}')]"
        element = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, xpath)))
        driver.execute_script("arguments[0].click();", element)
        time.sleep(0.5)
        return True
    except: return False

def load_data_from_sheet(sheet_name):
    print(f"📂 正在讀取工作表: {sheet_name} ...")
    try:
        sh = gc.open(sheet_name)
        worksheet = sh.sheet1
        data = worksheet.get_all_records()
        if not data: return {}

        df = pd.DataFrame(data)
        if '物件ID' not in df.columns: return {}

        data_map = {}
        for index, row in df.iterrows():
            pid = str(row['物件ID'])
            data_map[pid] = row.to_dict()
        print(f"✅ 成功載入 {len(data_map)} 筆紀錄。")
        return data_map
    except Exception as e:
        print(f"   -> 讀取失敗或是新檔案: {e}")
        return {}

def save_full_data(df, sheet_name):
    print(f"💾 正在儲存至: {sheet_name} ...")
    try:
        try: sh = gc.open(sheet_name)
        except: sh = gc.create(sheet_name)

        worksheet = sh.sheet1
        worksheet.clear()
        worksheet.append_row(df.columns.tolist())
        worksheet.append_rows(df.values.tolist())
        print(f"✅ 儲存完成！共 {len(df)} 筆。連結: {sh.url}")
    except Exception as e:
        print(f"❌ 儲存失敗: {e}")

# --- 3. 強效地理計算函式 (已修改為使用全域變數) ---
def get_distance_from_anchor(address_str):
    """
    輸入地址，回傳 (距離, 緯度, 經度)。
    使用全域變數 TARGET_CITY 來補全地址。
    """
    geolocator = Nominatim(user_agent="my_rental_scraper_custom_v1")

    # 使用設定檔中的城市名稱
    if TARGET_CITY not in address_str:
        full_addr = TARGET_CITY + address_str
    else:
        full_addr = address_str

    try:
        # 第一次嘗試：完整地址
        location = geolocator.geocode(full_addr, timeout=5)

        # 如果失敗 (None)，嘗試模糊化
        if not location:
            # 移除數字、號、樓、之，保留路名
            fuzzy_addr = re.sub(r'\d+[號樓之F].*', '', full_addr)
            if len(fuzzy_addr) > len(TARGET_CITY) + 1:
                location = geolocator.geocode(fuzzy_addr, timeout=5)

        if location:
            target_point = (location.latitude, location.longitude)
            distance = geodesic(ANCHOR_COORDS, target_point).km
            return round(distance, 2), location.latitude, location.longitude
        else:
            return "N/A", "N/A", "N/A"

    except Exception as e:
        return "N/A", "N/A", "N/A"

# 預設載入資料庫
history_database = load_data_from_sheet(SHEET_NAME_NEW)

In [ ]:
# @title 4. 執行爬蟲：全自動動態配置版 (Run to Scrape)
if 'history_database' not in globals():
    print("⚠️ 請先執行 Cell 3！")
else:
    print(f"🚀 啟動爬蟲 | 目標: {TARGET_CITY} {TARGET_DISTRICTS}...")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    # 1. 動態生成網址 (使用 TARGET_REGION_CODE)
    target_url = f"https://rental.example.com.tw/?region={TARGET_REGION_CODE}"
    driver.get(target_url)

    try:
        # A. 導航與篩選
        try:
            WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.close, i.close, .TIGerm"))).click()
        except: pass
        time.sleep(2)

        # --- 2. 動態點擊行政區 ---
        print("⚡ 正在設定篩選條件...")
        # 遍歷設定檔中的行政區列表進行點擊
        for district in TARGET_DISTRICTS:
            if click_element_by_text(driver, district):
                print(f"   -> 已選取: {district}")
            else:
                print(f"   ⚠️ 警告: 找不到或無法點擊 '{district}'")

        # 點擊租屋類型
        click_element_by_text(driver, RENTAL_TYPE)

        try:
            search_btn = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'搜尋')] | //div[contains(@class,'search')]//button")))
            driver.execute_script("arguments[0].click();", search_btn)
        except: pass

        print("⏳ 載入資料中...")
        time.sleep(5)
        for i in range(5):
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(1.5)

        # B. 解析
        items = driver.find_elements(By.CSS_SELECTOR, ".vue-list-rent-item, .listing-recommend-item, div[class*='item']")
        valid_items = [item for item in items if item.size['height'] > 80]

        current_data_list = []
        today_str = datetime.now(TW_TZ).strftime("%Y-%m-%d")
        last_check_time = datetime.now(TW_TZ).strftime("%Y-%m-%d %H:%M:%S")

        # --- 3. 動態生成行政區 Regex ---
        # 將列表 ["***HIDDEN_District***", "***HIDDEN_District***"] 轉成 "(***HIDDEN_District***|***HIDDEN_District***)"
        district_regex_str = f"({'|'.join(TARGET_DISTRICTS)})"

        for item in valid_items:
            try:
                full_text = item.text
                if len(full_text) < 10: continue

                # ID/連結過濾
                link = "N/A"
                try: link = item.find_element(By.TAG_NAME, "a").get_attribute("href")
                except: pass

                if "rental.example.com.tw" not in link: continue

                post_id = "N/A"
                id_match = re.search(r'(\d{7,8})', link)
                if id_match: post_id = id_match.group(1)
                if post_id == "N/A": continue

                # 價格/坪數
                price = "N/A"
                match = re.search(r'(\d{1,3}(,\d{3})*)\s*元/月', full_text)
                if match: price = match.group(0).replace("元/月", "").strip()
                else: continue

                area = "N/A"
                match = re.search(r'(\d+\.?\d*)\s*坪', full_text)
                if match: area = match.group(1)
                try:
                    if area != "N/A" and float(area) < 20: continue
                except: pass

                lines = full_text.split('\n')
                title = lines[0] if lines else "N/A"
                if len(title) < 5 and len(lines) > 1: title = lines[1]

                site_update_text = "N/A"
                for line in lines:
                    if any(k in line for k in ["更新", "發布", "前", "昨天", "剛"]):
                        if len(line) < 15:
                            site_update_text = line
                            break

                # --- 4. 地址解析 (使用全域變數 TARGET_CITY) ---
                location_full, city, district, address_road = "N/A", TARGET_CITY, "N/A", "N/A"
                for line in lines:
                    if "區" in line and ("市" in line or "-" in line):
                        location_full = line
                        break
                if location_full != "N/A":
                    parts = re.split(r'[-/ ]', location_full)
                    for p in parts:
                        if "區" in p: district = p
                        if "路" in p or "街" in p: address_road = p

                # 若抓不到區，使用動態生成的 Regex 再次搜尋
                if district == "N/A":
                    m = re.search(district_regex_str, full_text)
                    if m: district = m.group(1)

                # 5. 距離計算
                dist_km, lat, lon = "N/A", "N/A", "N/A"
                is_processed = False

                if post_id in history_database:
                    old = history_database[post_id]
                    if '距離(km)' in old and old['距離(km)'] != "N/A":
                        dist_km = old['距離(km)']
                        lat = old.get('緯度', "N/A")
                        lon = old.get('經度', "N/A")
                        is_processed = True

                if not is_processed and address_road != "N/A":
                    search_target = location_full if location_full != "N/A" else f"{district}{address_road}"
                    search_target = search_target.replace("-", "").replace("/", "")
                    dist_km, lat, lon = get_distance_from_anchor(search_target)
                    time.sleep(1.1)

                # 6. 歷史比對
                first_seen, days_mkt = today_str, 0
                if post_id in history_database:
                    old = history_database[post_id]
                    if '首次發現日' in old and old['首次發現日']:
                        first_seen = old['首次發現日']
                        try: days_mkt = (datetime.strptime(today_str, "%Y-%m-%d") - datetime.strptime(first_seen, "%Y-%m-%d")).days
                        except: pass

                current_data_list.append({
                    "物件ID": post_id,
                    "最後更新時間": last_check_time,
                    "首次發現日": first_seen,
                    "上架已持續天數": days_mkt,
                    "網站顯示更新": site_update_text,
                    "標題": title,
                    "價格": price,
                    "坪數": area,
                    "距離(km)": dist_km,
                    "緯度": lat,
                    "經度": lon,
                    "完整顯示地址": location_full,
                    "路段/地址": address_road,
                    "連結": link
                })

            except Exception: continue

        # C. 存檔
        if current_data_list:
            df_new = pd.DataFrame(current_data_list)
            df_new['廣告投放數'] = df_new.groupby(['價格', '坪數'])['物件ID'].transform('count')

            final_map = history_database.copy()
            for idx, row in df_new.iterrows():
                final_map[row['物件ID']] = row.to_dict()

            df_final = pd.DataFrame(list(final_map.values()))
            if '最後更新時間' in df_final.columns:
                df_final = df_final.sort_values(by='最後更新時間', ascending=False)

            cols = ["物件ID", "最後更新時間", "首次發現日", "上架已持續天數", "距離(km)", "網站顯示更新", "標題", "價格", "坪數", "廣告投放數", "路段/地址", "完整顯示地址", "緯度", "經度", "連結"]
            df_final = df_final[[c for c in cols if c in df_final.columns]]

            save_full_data(df_final, SHEET_NAME_NEW)
            history_database = final_map
        else:
            print("⚠️ 未抓取到資料。")

    except Exception as e:
        print(f"❌ 錯誤: {str(e)}")
        driver.quit()

    driver.quit()

---

### 如何執行你的「一週測試計畫」

既然你要測試一週，且每三天執行一次，操作流程如下：

1. **第一次（今天）：**
* 登入你的 Google Drive，建立一個 Colab 筆記本。
* 將上述三段代碼貼入。
* 依序點擊「播放鍵」執行 Cell 1, 2, 3, 4。
* 執行完後，去你的 Google Drive 根目錄找找看，會有一個 **`TARGET_SHEET_NAME`** 的試算表。打開來確認資料是否正確。


2. **第二次（三天後）：**
* 打開這個 Colab 網頁。
* **重要：** 因為 Colab 會重置環境，所以你必須**再次點擊 Cell 1, 2, 3, 4**。
* 程式會自動把新的資料「新增」到那張試算表的下面，不會覆蓋舊資料。


3. **第三次（六天後）：**
* 重複上述動作。



### 提醒（關於目標租屋網的反爬蟲）

在 Colab 的「無頭模式（Headless）」下，瀏覽器特徵非常明顯，租屋網這種網站有時會直接阻擋（你可能會看到程式跑完但說「抓到 0 筆物件」）。

* **如果發生這種情況**：代表目標租屋網擋掉了 Colab 的 IP 或特徵。這時候最簡單的解法，還是回到我一開始提供的 **PC 本地端執行**（因為你在本地有視窗介面，比較像真人）。

#### 第五步(Optional)：一次性資料遷移腳本 (One-time Migration)（Cell 5）

它會讀取 SHEET_NAME_OLD (備份檔)，針對每一筆資料檢查是否缺少距離。

如果缺少，呼叫我們新的強效地理計算函式補上，最後將補完的資料存入 SHEET_NAME_NEW。

執行完這次後，這個 Cell 就不需要再跑了。

In [ ]:
# @title 5. 資料遷移：補完舊資料的經緯度 (執行一次即可)

def migrate_old_data():
    print("🚀 開始執行舊資料遷移與補完計畫...")

    # 1. 讀取舊資料 (備份檔)
    old_map = load_data_from_sheet(SHEET_NAME_OLD)
    if not old_map:
        print("❌ 找不到舊資料備份，請確認 SHEET_NAME_OLD 設定是否正確。")
        return

    print(f"📦 讀取到 {len(old_map)} 筆舊資料，開始檢查缺失的經緯度...")

    updated_count = 0
    processed_list = []

    # 2. 遍歷舊資料
    for pid, row in old_map.items():
        # 檢查是否需要補距離
        need_update = False
        dist = str(row.get('距離(km)', 'N/A'))

        # 如果距離是 N/A 或者根本沒有這個欄位，就需要補算
        if dist == 'N/A' or dist == '':
            addr_full = str(row.get('完整顯示地址', ''))
            addr_road = str(row.get('路段/地址', ''))

            # 組合搜尋字串
            search_target = addr_full if len(addr_full) > 5 else f"高雄市{addr_road}"
            search_target = search_target.replace("-", "").replace("/", "")

            if len(search_target) > 3: # 確保有字可查
                print(f"   🔧 正在補算: {row.get('標題', 'Unknown')} ({search_target})...")
                dist_km, lat, lon = get_distance_from_anchor(search_target)

                # 更新欄位
                row['距離(km)'] = dist_km
                row['緯度'] = lat
                row['經度'] = lon

                updated_count += 1
                time.sleep(1.1) # 禮貌性延遲
            else:
                row['距離(km)'] = 'N/A'
                row['緯度'] = 'N/A'
                row['經度'] = 'N/A'

        processed_list.append(row)

    print(f"✨ 補完作業結束！共更新了 {updated_count} 筆資料。")

    # 3. 讀取新表目前的資料 (如果有的話)，避免覆蓋掉剛剛爬蟲剛抓的新資料
    #    但如果你的目的是「用舊資料初始化新表」，則直接存入即可。
    #    這裡我們採用「合併」策略：舊資料補完後 + 新表已有的資料

    new_map_current = load_data_from_sheet(SHEET_NAME_NEW)

    # 將補完的舊資料 merge 進去 (如果 ID 相同，以舊資料為主，因為我們要保留首次發現日)
    # 但通常我們希望保留最新的「最後更新時間」。
    # 策略：以 ID 為準，將 processed_list 轉回 map

    final_map = new_map_current.copy()
    for row in processed_list:
        pid = str(row['物件ID'])
        # 如果新表中已經有這筆，我們只更新它的經緯度 (避免覆蓋掉最新的價格或狀態)
        if pid in final_map:
            if row['距離(km)'] != 'N/A':
                final_map[pid]['距離(km)'] = row['距離(km)']
                final_map[pid]['緯度'] = row['緯度']
                final_map[pid]['經度'] = row['經度']
        else:
            # 如果新表沒這筆，直接加入
            final_map[pid] = row

    # 4. 存回新表
    df_final = pd.DataFrame(list(final_map.values()))

    # 排序
    if '最後更新時間' in df_final.columns:
        df_final = df_final.sort_values(by='最後更新時間', ascending=False)

    # 欄位整理
    cols = ["物件ID", "最後更新時間", "首次發現日", "上架已持續天數", "距離(km)", "網站顯示更新", "標題", "價格", "坪數", "廣告投放數", "路段/地址", "完整顯示地址", "緯度", "經度", "連結"]
    df_final = df_final[[c for c in cols if c in df_final.columns]]

    save_full_data(df_final, SHEET_NAME_NEW)

    # 更新全域變數，讓爬蟲知道
    global history_database
    history_database = final_map

# 執行遷移
migrate_old_data()